# 🌿 Phase 3 — Baseline Model Training (MobileNetV2)
**Project:** Plant Disease Detection System Using AI  
**Institution:** VIT Bhopal University  
**Supervisor:** Dr. Jagriti Gupta  

---

### 📌 Plan Overview
1. **Input Pipeline (`tf.data`)**: Load images from `train.csv` and `val.csv`, resize to 224×224, batch 32, prefetch.
2. **Data Augmentation**: Flips, rotations, zoom, and contrast changes on training images.
3. **Model Architecture**:
   - `Rescaling(1/127.5, offset=-1)` built directly inside the model so the model handles its own normalization.
   - Pretrained `MobileNetV2` backbone (ImageNet weights).
   - Global Average Pooling + Dropout(0.3) + Dense classification head.
4. **Two-Round Transfer Learning**:
   - **Round 1:** Freeze base model, train head with Adam (`lr=1e-3`) for up to 10 epochs.
   - **Round 2:** Unfreeze top 30 layers, fine-tune with Adam (`lr=1e-5`) for up to 15 epochs with `ReduceLROnPlateau`.
5. **Artifacts Saved**:
   - Best model: `models/plant_model_v1.keras`
   - Label index mapping: `models/labels.json`
   - Loss & accuracy curves: `docs/training_curves.png`
   - Experiment record: `docs/experiment_log.csv`

## 1. Setup & Environment Verification

In [ ]:
# Mount Google Drive if running on Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print('✅ Running on Google Colab — Google Drive mounted.')
except ImportError:
    IS_COLAB = False
    print('💻 Running locally.')

import os
import json
import csv
from datetime import datetime
from pathlib import Path

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score

print(f'TensorFlow Version : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'🚀 GPU Detected     : {gpus}')
else:
    print('⚠️  No GPU detected. Training will run on CPU (recommended: use Colab GPU for fast training).')

## 2. Configuration & Paths

In [ ]:
# Set project paths (adjust base path if running from Colab)
PROJECT_ROOT = Path('.') if not IS_COLAB else Path('/content/plant-detection-data-and-AI')

TRAIN_CSV = PROJECT_ROOT / 'data/splits/train.csv'
VAL_CSV   = PROJECT_ROOT / 'data/splits/val.csv'
MODELS_DIR = PROJECT_ROOT / 'models'
DOCS_DIR   = PROJECT_ROOT / 'docs'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
IMG_SIZE   = (224, 224)
SEED       = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
print('✅ Paths and hyperparameters configured.')

## 3. Load Splits & Build Class Labels Mapping

In [ ]:
assert TRAIN_CSV.exists(), f'Missing {TRAIN_CSV}. Complete Phase 2 splits first.'
assert VAL_CSV.exists(),   f'Missing {VAL_CSV}. Complete Phase 2 splits first.'

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

classes = sorted(train_df['label'].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}
idx_to_class = {idx: name for idx, name in enumerate(classes)}

train_df['target'] = train_df['label'].map(class_to_idx)
val_df['target']   = val_df['label'].map(class_to_idx)

num_classes = len(classes)
print(f'Total classes: {num_classes}')
print(f'Train samples: {len(train_df):,} | Val samples: {len(val_df):,}')

# Save labels.json mapping (class index -> class name)
labels_path = MODELS_DIR / 'labels.json'
with open(labels_path, 'w', encoding='utf-8') as f:
    json.dump(idx_to_class, f, indent=2)
print(f'✅ labels.json saved at: {labels_path}')

## 4. `tf.data` Input Pipeline & Augmentation

In [ ]:
def parse_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='augmentation')

# Build training pipeline (shuffled, augmented, batched, prefetched)
train_ds = tf.data.Dataset.from_tensor_slices((train_df['path'].values, train_df['target'].values))
train_ds = train_ds.shuffle(len(train_df), seed=SEED)
train_ds = train_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

# Build validation pipeline (no augmentation)
val_ds = tf.data.Dataset.from_tensor_slices((val_df['path'].values, val_df['target'].values))
val_ds = val_ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print('✅ tf.data pipelines assembled successfully.')

## 5. Model Architecture (MobileNetV2 + Built-in Rescaling)

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3), name='input_image')

# Rescaling layer inside the model: normalizes [0, 255] -> [-1, 1]
# This ensures exported TFLite model carries its own preprocessing!
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name='rescaling')(inputs)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
x = tf.keras.layers.Dropout(0.3, name='dropout')(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier_head')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='plant_disease_mobilenetv2')
model.summary()

## 6. Round 1 Training: Frozen Backbone

In [ ]:
# Calculate class weights for imbalance
class_counts = train_df['target'].value_counts().to_dict()
total_samples = len(train_df)
class_weights = {cls: total_samples / (num_classes * count) for cls, count in class_counts.items()}

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_r1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

print('🚀 Starting Round 1 Training (10 Epochs max)...')
history_r1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_r1
)

## 7. Round 2 Training: Fine-Tuning Top 30 Layers

In [ ]:
# Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

best_model_path = MODELS_DIR / 'plant_model_v1.keras'
callbacks_r2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(best_model_path), monitor='val_loss', save_best_only=True, verbose=1)
]

print('🚀 Starting Round 2 Fine-Tuning (15 Epochs max)...')
history_r2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_r2
)

## 8. Plot Curves & Record Experiment

In [ ]:
# Plot combined training curves
acc = history_r1.history['accuracy'] + history_r2.history['accuracy']
val_acc = history_r1.history['val_accuracy'] + history_r2.history['val_accuracy']
loss = history_r1.history['loss'] + history_r2.history['loss']
val_loss = history_r1.history['val_loss'] + history_r2.history['val_loss']
epochs_r1_len = len(history_r1.history['accuracy'])

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.axvline(x=epochs_r1_len - 1, color='gray', linestyle='--', label='Start Fine-Tuning')
plt.title('Accuracy Curve', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.axvline(x=epochs_r1_len - 1, color='gray', linestyle='--', label='Start Fine-Tuning')
plt.title('Loss Curve', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

curves_path = DOCS_DIR / 'training_curves.png'
plt.savefig(curves_path, dpi=120)
plt.show()
print(f'✅ Curves saved to: {curves_path}')

# Evaluate metrics
val_preds_probs = model.predict(val_ds)
val_preds = np.argmax(val_preds_probs, axis=1)
macro_f1 = f1_score(val_df['target'].values, val_preds, average='macro')
final_val_loss, final_val_acc = model.evaluate(val_ds, verbose=0)

# Log to experiment_log.csv
log_entry = {
    'run_id': f'run_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    'timestamp': datetime.now().isoformat(),
    'backbone': 'MobileNetV2',
    'image_size': '224x224',
    'batch_size': BATCH_SIZE,
    'epochs_round1': len(history_r1.history['accuracy']),
    'epochs_round2': len(history_r2.history['accuracy']),
    'optimizer': 'Adam',
    'lr_round1': '1e-3',
    'lr_round2': '1e-5',
    'val_accuracy': round(float(final_val_acc), 4),
    'val_loss': round(float(final_val_loss), 4),
    'val_macro_f1': round(float(macro_f1), 4),
    'model_path': str(best_model_path),
    'notes': 'MobileNetV2 transfer learning baseline with built-in Rescaling layer'
}

log_file = DOCS_DIR / 'experiment_log.csv'
df_log = pd.DataFrame([log_entry])
df_log.to_csv(log_file, mode='a', header=not log_file.exists(), index=False)
print(f'✅ Experiment logged to: {log_file}')